# Calidad de las bioactividades y filtro de promiscuidad

Composición de las bioactividades por tag y por fuente, y **filtro de
promiscuidad química**: 30 compuestos de `MW < 150 Da` concentran 104 075
relaciones de subestructura, el 12 % de la capa, y hoy ninguna está filtrada
(S3 Fig del paper 2016).

**Decidido (README §7.3): el filtro se aplica.** Este notebook produce
`03_compuestos_promiscuos.csv`, que `huerfanas/` consume vía
`tdr.compuestos_promiscuos()` antes de construir cualquier semilla. Por eso hay
que correrlo **antes** del paso 5 del orden de trabajo.

Fuente: `reproducir_v6/capa_quimica/02_filtro_promiscuidad.py`.

Salidas: `03_bioactividades_por_tag.csv`, `03_promiscuidad_por_compuesto.csv`,
`03_compuestos_promiscuos.csv`, `03_curva_filtrado.csv`, `03_impacto_filtro.csv`.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from pathlib import Path
# raiz del repositorio: se busca hacia arriba la carpeta que tiene DB/,
# asi el notebook corre desde donde sea que se haya clonado
RAIZ = Path.cwd()
while not (RAIZ / "DB").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "comun"))
sys.path.insert(0, str(RAIZ / "analiceDB"))
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_analice as fa              # el .py de esta carpeta

SALIDAS = tdr.out("analiceDB")
FIGURAS = SALIDAS / "figuras"
NB      = "03"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
datos = tdr.cargar_db(anotaciones=False, cluster_consistent=False)
# Las relaciones de subestructura crudas: la capa guardada ya está clusterizada
# y no conserva el conteo de superestructuras por molécula.
sub, comp = fa.cargar_subestructuras_crudas()
len(datos.bioact), len(sub), len(comp)

## Acondicionamiento

In [ ]:
tags = fa.composicion_bioactividades(datos)
print(f"{len(sub)} relaciones de subestructura mapeadas a compuestos")
print(f"criterio: MW < {tdr.MW_PROMISCUIDAD} Da y N_parentales > {tdr.N_PARENTALES_PROMISCUIDAD}")
tags.head(12)

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
por_compuesto, promiscuos, impacto = fa.promiscuidad_subestructural(sub, comp)
curva = fa.curva_filtrado(por_compuesto, sub)

tags.to_csv(SALIDAS / f"{NB}_bioactividades_por_tag.csv", index=False)
por_compuesto.to_csv(SALIDAS / f"{NB}_promiscuidad_por_compuesto.csv", index=False)
promiscuos.to_csv(SALIDAS / f"{NB}_compuestos_promiscuos.csv", index=False)  # <- lo lee huerfanas/
curva.to_csv(SALIDAS / f"{NB}_curva_filtrado.csv", index=False)
impacto.to_csv(SALIDAS / f"{NB}_impacto_filtro.csv", index=False)
fa.escribir_meta(SALIDAS, NB, notebook="03_calidad_bioactividades.ipynb",
                 params={"mw_max": tdr.MW_PROMISCUIDAD,
                         "n_parentales_min": tdr.N_PARENTALES_PROMISCUIDAD},
                 aplica_filtro=True, consumido_por="huerfanas/")

# Resultados

In [ ]:
tags          = pd.read_csv(SALIDAS / f"{NB}_bioactividades_por_tag.csv")
por_compuesto = pd.read_csv(SALIDAS / f"{NB}_promiscuidad_por_compuesto.csv")
promiscuos    = pd.read_csv(SALIDAS / f"{NB}_compuestos_promiscuos.csv")
curva         = pd.read_csv(SALIDAS / f"{NB}_curva_filtrado.csv")
impacto       = pd.read_csv(SALIDAS / f"{NB}_impacto_filtro.csv")
impacto.T

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), tight_layout=True)
t = tags.groupby("activity_tag")["n"].sum().sort_values()
ax.barh(range(len(t)), t.values, color=tdr.S1)
ax.set_yticks(range(len(t)))
# los tags se muestran con su etiqueta, no con el codigo
ax.set_yticklabels([tdr.TAG_NOMBRE.get(i, i) for i in t.index], fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("registros compuesto-proteína")
ax.set_title("Composición por tag de actividad")
tdr.guardar(fig, f"{NB}_f01_bioactividades_por_tag", FIGURAS)

In [ ]:
fig = fa.fig_promiscuidad(por_compuesto, curva, plt)
tdr.guardar(fig, f"{NB}_f02_promiscuidad", FIGURAS)

In [ ]:
# Los compuestos que se van: es el resultado que condiciona todo lo que viene después
promiscuos.head(30)